<a href="https://colab.research.google.com/github/emily-speckman/SinkholeProject/blob/main/AZKarstDL_01_create_geopackage_grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AZKarstDL_01_create_geopackage_grid

This notebook creates the foundational tile grid for the Arizona Karst Deep Learning project. It takes a boundary shapefile of the State of Arizona, reprojects it to NAD83(2011) / UTM Zone 12N, and generates a statewide 20 km × 20 km square tile grid snapped to a fixed 10 m-aligned origin (0,0) in UTM coordinates. The notebook exports a GeoPackage containing (1) the Arizona boundary, (2) the full square tiles that intersect Arizona (used for DEM downloads and model tiling), and optionally (3) tiles clipped to the state boundary for visualization. This grid becomes the spatial backbone for all data acquisition, preprocessing, model training, and statewide inference.


1. Mount Drive + install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install geopandas shapely pyproj fiona rtree folium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 741.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 51.6 MB/s eta 0:00:00


2. Point to your AZ boundary shapefile

  *   Open shapefile of Arizona
  *   Load it



In [ ]:
import geopandas as gpd
import folium

AZ_ZIP = "/content/drive/MyDrive/DATA_COLAB/AZ_Census_State_Bound.zip"

# Read shapefile from zip
az = gpd.read_file(f"zip://{AZ_ZIP}")

print("CRS:", az.crs)
print("Features:", len(az))
az.head()


DataSourceError: '/vsizip//content/drive/MyDrive/DATA_COLAB/AZ_Census_State_Bound.zip' does not exist in the file system, and is not recognized as a supported dataset name.

3. Reproject to EPSG:26912 and dissolve to one polygon

In [ ]:
TARGET_CRS = "EPSG:26912"  # NAD83(2011) / UTM Zone 12N

az_utm = az.to_crs(TARGET_CRS)

# Dissolve to a single geometry (state outline)
az_union = az_utm.dissolve()  # keeps 1 row
az_geom = az_union.geometry.iloc[0]

az_union


NameError: name 'az' is not defined

4. Build 20 km grid snapped to origin (0,0)
  
  This creates tile IDs like AZ_UTM12N_20km_R{row}_C{col} where row/col are based on 20,000 m bins from origin.

In [ ]:
import math
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

TILE_SIZE_M = 20000  # 20 km
x_min, y_min, x_max, y_max = az_union.total_bounds

# Snap bbox outward to tile boundaries (origin = 0,0)
col_min = math.floor(x_min / TILE_SIZE_M)
col_max = math.ceil(x_max / TILE_SIZE_M) - 1
row_min = math.floor(y_min / TILE_SIZE_M)
row_max = math.ceil(y_max / TILE_SIZE_M) - 1

records = []
geoms = []

for r in range(row_min, row_max + 1):
    for c in range(col_min, col_max + 1):
        xmin = c * TILE_SIZE_M
        xmax = xmin + TILE_SIZE_M
        ymin = r * TILE_SIZE_M
        ymax = ymin + TILE_SIZE_M

        geom = box(xmin, ymin, xmax, ymax)
        tile_id = f"AZ_UTM12N_20km_R{r}_C{c}"

        records.append({
            "tile_id": tile_id,
            "row": r,
            "col": c,
            "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax
        })
        geoms.append(geom)

grid_full = gpd.GeoDataFrame(pd.DataFrame(records), geometry=gpd.GeoSeries(geoms, crs=TARGET_CRS), crs=TARGET_CRS)

print("Full grid tiles:", len(grid_full))
grid_full.head()


Full grid tiles: 896


,tile_id,row,col,xmin,ymin,xmax,ymax,geometry
0,AZ_UTM12N_20km_R173_C7,173,7,140000,3460000,160000,3480000,"POLYGON ((160000 3460000, 160000 3480000, 1400..."
1,AZ_UTM12N_20km_R173_C8,173,8,160000,3460000,180000,3480000,"POLYGON ((180000 3460000, 180000 3480000, 1600..."
2,AZ_UTM12N_20km_R173_C9,173,9,180000,3460000,200000,3480000,"POLYGON ((200000 3460000, 200000 3480000, 1800..."
3,AZ_UTM12N_20km_R173_C10,173,10,200000,3460000,220000,3480000,"POLYGON ((220000 3460000, 220000 3480000, 2000..."
4,AZ_UTM12N_20km_R173_C11,173,11,220000,3460000,240000,3480000,"POLYGON ((240000 3460000, 240000 3480000, 2200..."


5. Keep only tiles that intersect Arizona + clip geometry

  This produces:

    grid_tiles: full tile squares that intersect AZ (good for raster tiling)

    grid_clipped: tile polygons clipped to the AZ boundary (good for visualization / summaries)

In [ ]:
# Spatial index speeds this up
grid_tiles = grid_full[grid_full.intersects(az_geom)].copy()
grid_tiles["in_az"] = True

# Optional: clipped geometry (tile polygon ∩ AZ polygon)
grid_clipped = grid_tiles.copy()
grid_clipped["geometry"] = grid_clipped.geometry.intersection(az_geom)

print("Intersecting tiles:", len(grid_tiles))
grid_tiles.head()


Intersecting tiles: 794


,tile_id,row,col,xmin,ymin,xmax,ymax,geometry,in_az
15,AZ_UTM12N_20km_R173_C22,173,22,440000,3460000,460000,3480000,"POLYGON ((460000 3460000, 460000 3480000, 4400...",True
16,AZ_UTM12N_20km_R173_C23,173,23,460000,3460000,480000,3480000,"POLYGON ((480000 3460000, 480000 3480000, 4600...",True
17,AZ_UTM12N_20km_R173_C24,173,24,480000,3460000,500000,3480000,"POLYGON ((500000 3460000, 500000 3480000, 4800...",True
18,AZ_UTM12N_20km_R173_C25,173,25,500000,3460000,520000,3480000,"POLYGON ((520000 3460000, 520000 3480000, 5000...",True
19,AZ_UTM12N_20km_R173_C26,173,26,520000,3460000,540000,3480000,"POLYGON ((540000 3460000, 540000 3480000, 5200...",True


6. Export to GeoPackage in Drive

You’ll get one .gpkg with multiple layers:

    az_boundary

    tilegrid_20km_full_intersect (full tiles that intersect AZ)

    tilegrid_20km_clipped (clipped to AZ outline)


> Note:For your DEM download + model inference, you almost always want to use:

>tilegrid_20km_full_intersect (full squares)

>Because your rasters/patch windows want consistent rectangular extents. The >clipped layer is mainly for cartography.

In [ ]:
OUT_GPKG = "/content/drive/MyDrive/DATA_COLAB/az_tilegrid_20km_utm12n.gpkg"

# Write layers
az_union.to_file(OUT_GPKG, layer="az_boundary", driver="GPKG")
grid_tiles.to_file(OUT_GPKG, layer="tilegrid_20km_full_intersect", driver="GPKG")
grid_clipped.to_file(OUT_GPKG, layer="tilegrid_20km_clipped", driver="GPKG")

print("Wrote:", OUT_GPKG)


Wrote: /content/drive/MyDrive/DATA_COLAB/az_tilegrid_20km_utm12n.gpkg


7. Plot the new grid to visualize

In [ ]:

import folium

# Paths
GRID_GPKG = "/content/drive/MyDrive/DATA_COLAB/az_tilegrid_20km_utm12n.gpkg"

# Load layers
az_boundary = gpd.read_file(GRID_GPKG, layer="az_boundary")
grid_tiles = gpd.read_file(GRID_GPKG, layer="tilegrid_20km_full_intersect")

# Convert to WGS84 for web mapping
az_boundary_wgs = az_boundary.to_crs("EPSG:4326")
grid_tiles_wgs = grid_tiles.to_crs("EPSG:4326")

# Get map center
center = az_boundary_wgs.geometry.centroid.iloc[0]
m = folium.Map(location=[center.y, center.x], zoom_start=6, tiles="OpenStreetMap")

# Add Arizona boundary
folium.GeoJson(
    az_boundary_wgs,
    name="Arizona Boundary",
    style_function=lambda x: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0
    }
).add_to(m)

# Add tile grid
folium.GeoJson(
    grid_tiles_wgs,
    name="20km Tile Grid",
    style_function=lambda x: {
        "color": "blue",
        "weight": 1,
        "fillOpacity": 0
    },
    tooltip=folium.GeoJsonTooltip(fields=["tile_id"])
).add_to(m)

folium.LayerControl().add_to(m)

m


/tmp/ipykernel_41393/111693588.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = az_boundary_wgs.geometry.centroid.iloc[0]


# Next steps

** In the next notebook we will grab tiles from areas of known sink holes for training our deep learning model. We will use the following tiles (areas)**

next notebook: `AZKarstDL_02_get_training_tile_study_areas.ipynb`

* AZ_UTM12N_20km_R201_C19 = North Kaibab Demonte Park and West
* AZ_UTM12N_20km_R198_C19 = Anita Karst near Woodin AZ
* AZ_UTM12N_20km_R189_C25 = South of Woods Canyon Lake Apache Sitgreives
* AZ_UTM12N_20km_R192_C26 = Chevlon drainage including McCauley Sinks
* AZ_UTM12N_20km_R191_C28 = Sinks SE of Zeniff
* AZ_UTM12N_20km_R191_C27 = contains most of the sinks NW of Zeniff





